# `expansion_hunter_00_prep_reference` — assembly38 FASTA + catalogs

Run **in the ExpansionHunter VWB JupyterLab app** before
[`expansion_hunter_01_run.ipynb`](expansion_hunter_01_run.ipynb). This VM does
not share the Locityper app's local disk, so restage anything EH needs here.

EH needs an **uncompressed** GATK
[`Homo_sapiens_assembly38.fasta`](https://support.researchallofus.org/hc/en-us/articles/4807740201876-What-reference-are-the-variants-called-against-for-the-genomic-data)
+ `.fai` (the FASTA AoU srWGS CRAMs are encoded against) and a variant catalog
JSON. It does **not** need Jellyfish or a Locityper `vcf_db`.

AoU v9 short-read CRAMs are listed at

`workspace/vwb-aou-datasets-controlled-v9/v9/wgs/cram/manifest.csv`

(`person_id`, `cram_uri`, `cram_index_uri` under
`gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/`). This notebook
does **not** download CRAMs; the minicram WDL reads them in place.

## What this notebook does

1. Find the CRAM manifest (smoke-test `person_id`)
2. Reuse `locityper/refs/` assembly38 on the workspace bucket if it is already
   there (the other VM may have uploaded it); otherwise copy the public Broad
   FASTA + fai into `expansion_hunter/refs/` (GCS-to-GCS, no 3 GB local copy)
3. Upload smoke + 711-locus catalogs from this repo
4. Print `expansion_hunter_01` config assignments

**Re-run is safe.** GCS objects and optional local files skip when the target
already exists. Set `FORCE_STAGE_REF`, `FORCE_UPLOAD`, or `DOWNLOAD_LOCAL` to
replace / pull a local FASTA.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


def capture(cmd: list[str], *, check: bool = True) -> str:
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, check=check, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n", file=sys.stderr)
    return proc.stdout


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        wdl = p / "expansion_hunter" / "wdl" / "ExpansionHunterMinicram.wdl"
        if wdl.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "expansion_hunter" / "wdl" / "ExpansionHunterMinicram.wdl"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError(
        "Cannot find expansion_hunter/wdl/ExpansionHunterMinicram.wdl. Clone "
        "kvg/aou-lr-phase-2 into this app (or cd into the clone) and re-run."
    )


def find_manifest() -> Path | None:
    env = (
        os.environ.get("EH_CRAM_MANIFEST", "").strip()
        or os.environ.get("LOCITYPER_CRAM_MANIFEST", "").strip()
    )
    if env:
        p = Path(env)
        if p.is_file():
            return p.resolve()
    here = Path.cwd().resolve()
    names = [
        Path("workspace/vwb-aou-datasets-controlled-v9/v9/wgs/cram/manifest.csv"),
        Path("v9/wgs/cram/manifest.csv"),
    ]
    roots = [here, *here.parents, Path("/home/jupyter"), Path("/home/jupyter/workspace")]
    for root in roots:
        for rel in names:
            cand = (root / rel) if not rel.is_absolute() else rel
            if cand.is_file():
                return cand.resolve()
    ws = Path("workspace")
    if ws.is_dir():
        hits = list(ws.glob("**/v9/wgs/cram/manifest.csv"))
        if hits:
            return hits[0].resolve()
    return None


def gcs_exists(uri: str) -> bool:
    proc = subprocess.run(
        ["gcloud", "storage", "objects", "describe", uri],
        check=False,
        capture_output=True,
        text=True,
    )
    return proc.returncode == 0


def gcs_cp(src: str | Path, dest: str | Path) -> None:
    cmd = ["gcloud", "storage", "cp", str(src), str(dest)]
    project = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if project:
        cmd.insert(3, f"--billing-project={project}")
    sh(cmd)


REPO_ROOT = find_repo_root()
SMOKE_CATALOG_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "smoke.catalog.json"
FULL_CATALOG_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "candidate_EH_Loci.GRCh38.json"
SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch" / "refs"
SCRATCH.mkdir(parents=True, exist_ok=True)

REF_STEM = "Homo_sapiens_assembly38"
REF_FA_NAME = f"{REF_STEM}.fasta"
REF_FAI_NAME = f"{REF_FA_NAME}.fai"
REF_FA_GCS_CANDIDATES = [
    "gs://gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta",
    "gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.fasta",
]
REF_FAI_GCS_CANDIDATES = [
    "gs://gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai",
    "gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.fasta.fai",
]
REF_FA_GCS = os.environ.get("EH_REF_FA_GCS", REF_FA_GCS_CANDIDATES[0])
REF_FAI_GCS = os.environ.get("EH_REF_FAI_GCS", REF_FAI_GCS_CANDIDATES[0])

OUTPUT_BUCKET_ID = os.environ.get("EH_OUTPUT_BUCKET_ID", "aou-lr-phase2-resources")
OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
STAGE_PREFIX = os.environ.get("EH_STAGE_PREFIX", "expansion_hunter")
REFS_PREFIX = os.environ.get("EH_REFS_PREFIX", f"{STAGE_PREFIX.strip('/')}/refs")
LOCITYPER_REFS_PREFIX = os.environ.get("EH_LOCITYPER_REFS_PREFIX", "locityper/refs")

SMOKE_PERSON_ID = os.environ.get("EH_SMOKE_PERSON_ID", "1000000")

FORCE_STAGE_REF = False
FORCE_UPLOAD = False
DOWNLOAD_LOCAL = False  # True: also pull FASTA onto this VM (not required for the WDL)
FORCE_FAIDX = False
CONDA_INSTALL = True  # samtools only if DOWNLOAD_LOCAL / FORCE_FAIDX needs it
UPLOAD = True

REF_FA = SCRATCH / REF_FA_NAME
REF_FAI = SCRATCH / REF_FAI_NAME

print("REPO_ROOT:", REPO_ROOT)
print("SCRATCH:", SCRATCH)
print("disk free GiB:", round(shutil.disk_usage(SCRATCH).free / 1024**3, 1))
print("OUTPUT_BUCKET_ID:", OUTPUT_BUCKET_ID or "(unset)")
print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS or "(unset)")
print("STAGE_PREFIX:", STAGE_PREFIX)
print("FORCE_STAGE_REF:", FORCE_STAGE_REF, "FORCE_UPLOAD:", FORCE_UPLOAD)
print("DOWNLOAD_LOCAL:", DOWNLOAD_LOCAL)
print("smoke catalog:", SMOKE_CATALOG_LOCAL.is_file(), SMOKE_CATALOG_LOCAL)
print("full catalog:", FULL_CATALOG_LOCAL.is_file(), FULL_CATALOG_LOCAL)
print("samtools:", shutil.which("samtools"))


## CRAM manifest

v9 WGS CRAMs (Illumina, `v8_base`). Use a `person_id` from this table as
`SAMPLE_ID` in the run notebook; pass `cram_uri` / `cram_index_uri` as
`CRAM` / `CRAI`.


In [ ]:
MANIFEST = find_manifest()
print("MANIFEST:", MANIFEST or "(not found — set EH_CRAM_MANIFEST)")

SMOKE_CRAM = ""
SMOKE_CRAI = ""
if MANIFEST is not None:
    with MANIFEST.open(newline="") as fh:
        rows = list(csv.DictReader(fh))
    print("n samples:", len(rows))
    print("columns:", list(rows[0].keys()) if rows else [])
    hit = next((r for r in rows if r.get("person_id") == str(SMOKE_PERSON_ID)), None)
    if hit is None and rows:
        hit = rows[0]
        SMOKE_PERSON_ID = hit["person_id"]
        print("SMOKE_PERSON_ID not in manifest; using first row", SMOKE_PERSON_ID)
    if hit:
        SMOKE_CRAM = hit["cram_uri"]
        SMOKE_CRAI = hit["cram_index_uri"]
        print("smoke person_id:", SMOKE_PERSON_ID)
        print("cram:", SMOKE_CRAM)
        print("crai:", SMOKE_CRAI)
    print("--- head ---")
    for row in rows[:5]:
        print(row["person_id"], row["cram_uri"])


## Workspace bucket

Uploads go to `OUTPUT_BUCKET_GS / STAGE_PREFIX`. `--output-bucket-id` in the
run notebook is this resource ID, not the `gs://` name.


In [ ]:
if shutil.which("wb"):
    sh(["wb", "auth", "status"], check=False)
    sh(["wb", "resource", "list", "--type=GCS_BUCKET"], check=False)
    if OUTPUT_BUCKET_ID and not OUTPUT_BUCKET_GS:
        resolved = capture(
            ["wb", "resource", "resolve", f"--id={OUTPUT_BUCKET_ID}"],
            check=False,
        ).strip().splitlines()
        resolved = next((line.strip() for line in reversed(resolved) if line.strip()), "")
        if resolved.startswith("gs://") or resolved.startswith("s3://"):
            OUTPUT_BUCKET_GS = resolved
        elif resolved and " " not in resolved:
            OUTPUT_BUCKET_GS = f"gs://{resolved}"
else:
    print("wb not on PATH; set OUTPUT_BUCKET_GS to a gs:// prefix you can write")

print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS or "(unset)")
if not OUTPUT_BUCKET_GS:
    print("Uploads will be skipped until OUTPUT_BUCKET_ID/GS is set.")


## Stage assembly38 FASTA + fai

Prefer an object that already exists on the workspace bucket:

1. `expansion_hunter/refs/Homo_sapiens_assembly38.fasta`
2. `locityper/refs/…` (the Locityper VM may have uploaded this already)
3. Otherwise copy the public Broad FASTA **GCS-to-GCS** into
   `expansion_hunter/refs/` (no local 3 GB download)

`FORCE_STAGE_REF=True` recopies from Broad into `expansion_hunter/refs/` even
if something is already there. Set `DOWNLOAD_LOCAL=True` only if you want a
copy on this VM (needed solely for a local `samtools faidx` fallback).


In [ ]:
def gcs_cp_first(uris: list[str], dest: str) -> str:
    last_err: Exception | None = None
    for uri in uris:
        try:
            gcs_cp(uri, dest)
            return uri
        except subprocess.CalledProcessError as exc:
            print("failed", uri)
            last_err = exc
    raise SystemExit(f"could not copy to {dest}: {last_err}") from last_err


if not OUTPUT_BUCKET_GS:
    raise SystemExit("Set OUTPUT_BUCKET_ID (and re-run the bucket cell) before staging the FASTA.")

bucket = OUTPUT_BUCKET_GS.rstrip("/")
eh_refs = f"{bucket}/{REFS_PREFIX.strip('/')}"
locityper_refs = f"{bucket}/{LOCITYPER_REFS_PREFIX.strip('/')}"

EH_REF_FA_GS = f"{eh_refs}/{REF_FA_NAME}"
EH_REF_FAI_GS = f"{eh_refs}/{REF_FAI_NAME}"
LOCITYPER_REF_FA_GS = f"{locityper_refs}/{REF_FA_NAME}"
LOCITYPER_REF_FAI_GS = f"{locityper_refs}/{REF_FAI_NAME}"

REF_FA_GS = ""
REF_FAI_GS = ""


def pair_ready(fa: str, fai: str) -> bool:
    return gcs_exists(fa) and gcs_exists(fai)


if not FORCE_STAGE_REF and pair_ready(EH_REF_FA_GS, EH_REF_FAI_GS):
    REF_FA_GS, REF_FAI_GS = EH_REF_FA_GS, EH_REF_FAI_GS
    print("skip FASTA stage; exists under expansion_hunter/refs/")
elif not FORCE_STAGE_REF and pair_ready(LOCITYPER_REF_FA_GS, LOCITYPER_REF_FAI_GS):
    REF_FA_GS, REF_FAI_GS = LOCITYPER_REF_FA_GS, LOCITYPER_REF_FAI_GS
    print("reuse Locityper-staged assembly38 (other VM already uploaded it)")
else:
    fa_uris = [REF_FA_GCS, *([u for u in REF_FA_GCS_CANDIDATES if u != REF_FA_GCS])]
    fai_uris = [REF_FAI_GCS, *([u for u in REF_FAI_GCS_CANDIDATES if u != REF_FAI_GCS])]
    used_fa = gcs_cp_first(fa_uris, EH_REF_FA_GS)
    print("staged FASTA from", used_fa)
    used_fai = gcs_cp_first(fai_uris, EH_REF_FAI_GS)
    print("staged fai from", used_fai)
    REF_FA_GS, REF_FAI_GS = EH_REF_FA_GS, EH_REF_FAI_GS

print("REF_FA:", REF_FA_GS)
print("REF_FAI:", REF_FAI_GS)


## Optional local FASTA

Skip unless `DOWNLOAD_LOCAL=True` or `FORCE_FAIDX=True`. The WDL localizes the
GCS FASTA itself.


In [ ]:
if DOWNLOAD_LOCAL or FORCE_FAIDX:
    need_fa = FORCE_STAGE_REF or not REF_FA.is_file()
    if need_fa:
        REF_FA.unlink(missing_ok=True)
        gcs_cp(REF_FA_GS, REF_FA)
        print("local FASTA", REF_FA, "bytes", REF_FA.stat().st_size)
    else:
        print("skip local FASTA; exists", REF_FA)

    need_fai = FORCE_STAGE_REF or FORCE_FAIDX or not REF_FAI.is_file()
    if need_fai:
        REF_FAI.unlink(missing_ok=True)
        try:
            gcs_cp(REF_FAI_GS, REF_FAI)
            print("local fai", REF_FAI)
        except subprocess.CalledProcessError:
            print("GCS fai copy failed; will samtools faidx")
    else:
        print("skip local fai; exists", REF_FAI)

    if FORCE_FAIDX or not REF_FAI.is_file():
        def prepend_bin_dirs() -> None:
            extras: list[str] = []
            for d in (
                Path(sys.prefix) / "bin",
                Path("/opt/conda/envs/jupyter/bin"),
                Path("/opt/conda/bin"),
            ):
                if d.is_dir():
                    extras.append(str(d.resolve()))
            os.environ["PATH"] = os.pathsep.join(extras + [os.environ.get("PATH", "")])

        prepend_bin_dirs()
        if shutil.which("samtools") is None and CONDA_INSTALL:
            exe = shutil.which("mamba") or shutil.which("conda")
            if exe is None:
                raise SystemExit("samtools missing and conda/mamba not on PATH")
            sh(
                [
                    exe,
                    "install",
                    "-y",
                    "-p",
                    sys.prefix,
                    "-c",
                    "bioconda",
                    "-c",
                    "conda-forge",
                    "samtools",
                ]
            )
            prepend_bin_dirs()
        samtools = shutil.which("samtools")
        if samtools is None:
            raise SystemExit("samtools is not on PATH; cannot faidx")
        sh([samtools, "faidx", str(REF_FA)])
        if UPLOAD and OUTPUT_BUCKET_GS and (FORCE_UPLOAD or not gcs_exists(EH_REF_FAI_GS)):
            gcs_cp(REF_FAI, EH_REF_FAI_GS)
            REF_FAI_GS = EH_REF_FAI_GS
            print("uploaded faidx", REF_FAI_GS)
    print("--- .fai head ---")
    print("\n".join(REF_FAI.read_text().splitlines()[:8]))
else:
    print("DOWNLOAD_LOCAL=False; WDL will read", REF_FA_GS)


## Upload catalogs

Smoke (two chr1 loci) and the 711-locus panel from
[`expansion_hunter/configs/`](../../expansion_hunter/configs/). Skip each
object if it already exists unless `FORCE_UPLOAD=True`.


In [ ]:
if not SMOKE_CATALOG_LOCAL.is_file() or not FULL_CATALOG_LOCAL.is_file():
    raise SystemExit(f"missing catalog JSON under {REPO_ROOT / 'expansion_hunter' / 'configs'}")

for path in (SMOKE_CATALOG_LOCAL, FULL_CATALOG_LOCAL):
    json.loads(path.read_text())

SMOKE_CATALOG_GS = ""
FULL_CATALOG_GS = ""

if UPLOAD:
    if not OUTPUT_BUCKET_GS:
        raise SystemExit("Set OUTPUT_BUCKET_ID (and re-run the bucket cell) before uploading catalogs.")
    prefix = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX.strip('/')}"
    SMOKE_CATALOG_GS = f"{prefix}/{SMOKE_CATALOG_LOCAL.name}"
    FULL_CATALOG_GS = f"{prefix}/{FULL_CATALOG_LOCAL.name}"
    for src, dest in (
        (SMOKE_CATALOG_LOCAL, SMOKE_CATALOG_GS),
        (FULL_CATALOG_LOCAL, FULL_CATALOG_GS),
    ):
        if not FORCE_UPLOAD and gcs_exists(dest):
            print("skip upload; exists", dest)
            continue
        gcs_cp(src, dest)
        print("staged catalog:", dest)
else:
    print("UPLOAD=False; catalogs stay in the git clone")
    SMOKE_CATALOG_GS = str(SMOKE_CATALOG_LOCAL)
    FULL_CATALOG_GS = str(FULL_CATALOG_LOCAL)


## Print run-notebook assignments

Paste these into `expansion_hunter_01_run` (or confirm the config-cell defaults
already match).


In [ ]:
if not SMOKE_CRAM:
    SMOKE_CRAM = f"gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_{SMOKE_PERSON_ID}.cram"
    SMOKE_CRAI = SMOKE_CRAM + ".crai"
    print("manifest miss; using conventional v8_base URIs")

print(
    f"""
Paste into expansion_hunter_01_run config:

SAMPLE_ID = "{SMOKE_PERSON_ID}"
CRAM = "{SMOKE_CRAM}"
CRAI = "{SMOKE_CRAI}"
REF_FA = "{REF_FA_GS}"
REF_FAI = "{REF_FAI_GS}"
CATALOG = "{SMOKE_CATALOG_GS}"
SEX = "female"

Full 711-locus catalog (swap CATALOG when smoke works):
  {FULL_CATALOG_GS}
"""
)
